# Chapter 10 — Rough, Temporal, and Fuzzy Modelling
### Notebook 3 · Exercises

*Book reference: Section 10.3*

The book's exercises, executable. Assertions are the marking scheme.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch10_toolkit as ch10
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

### Exercise R1 — Name the relation for six interval pairs

Give the Allen relation for each pair, and its inverse.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
pairs = [((0, 2), (3, 5)), ((0, 3), (3, 6)), ((0, 5), (2, 3)),
         ((1, 4), (1, 6)), ((2, 6), (1, 6)), ((1, 4), (1, 4))]
rows = []
for a, b in pairs:
    r = ch10.relation_between(a, b)
    back = ch10.relation_between(b, a)
    rows.append({'A': str(a), 'B': str(b), 'A r B': ch10.ALLEN_NAMES[r],
                 'B r A': ch10.ALLEN_NAMES[back],
                 'inverse holds': ch10.inverse(r) == back})
print(pd.DataFrame(rows).to_string(index=False))
assert all(row['inverse holds'] for row in rows)
print('\nThe inverse relation always holds in the other direction -- a property\n'
      'worth testing, since it is easy to get wrong when hand-writing the table.')

### Exercise R2 — Model 'a warm room' three ways

Give a crisp, a fuzzy and an alpha-cut treatment of the same requirement, and say what each one costs.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
temperatures = list(range(10, 36))
warm = ch10.trapezoid(16, 21, 26, 31)
print('crisp (>= 21):', [t for t in temperatures if t >= 21][:6], '...')
print('fuzzy degrees:')
for t in [18, 20, 22, 26, 30]:
    print(f'   {t}C -> {round(warm(t), 3)}')
alpha_half = [t for t in temperatures if warm(t) >= 0.5]
print('alpha-cut 0.5:', f'{min(alpha_half)}-{max(alpha_half)}C')
assert warm(18) < warm(20) < warm(22) == warm(26) == 1.0   # rises, then plateaus
print('\ncrisp     : one number, indefensible at the boundary, free to reason with.')
print('fuzzy     : honest about the gradient, needs a t-norm choice to combine.')
print('alpha-cut : back to a number, but one you had to state and can defend.')

### Exercise R3 — Report what your data cannot decide

For a target set, produce the report a clinician could act on: what is certain, what is possible, and what more data would resolve.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
system = ch10.SAMPLE_SYSTEM
target = ['p1', 'p3', 'p5']
print('certainly in the group :', system.lower_approximation(target))
print('possibly in the group  :', system.upper_approximation(target))
print('undecidable from data  :', system.boundary(target))
print('accuracy               :', system.accuracy(target))
print()
for group in system.indiscernibility():
    if len(group) > 1 and any(o in target for o in group) \
            and not set(group) <= set(target):
        print(f'  {group} are indiscernible but disagree on membership')
assert system.accuracy(target) < 1.0
print('\nThat last line is the actionable part: it names the exact pairs whose\n'
      'separation would raise accuracy, which turns "collect more data" into a\n'
      'specific request.')

## Where this leaves you

Three formalisms, each with a computable core and a known price. Notebook 4 asks an agent to choose between them — and prices the *search* that temporal reasoning requires.